In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from tqdm import tqdm
from afno import FourierTransformerUNet
PROCESSED_DATA_PATH = '../data/processed/flickr30k_256x256_augmented'
DEBUG_DATA_PATH = os.path.join(PROCESSED_DATA_PATH, 'debug')
ILLUMINATION = 'vortex'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hyperparameters
BATCH_SIZE = 5     # Small batch size for a small dataset (15 images / 5 = 3 batches)
LEARNING_RATE = 1e-4
EPOCHS = 8        # Run for enough epochs to see the loss decrease

# Model Parameters
IMG_SIZE = 256
PATCH_SIZE = 4
EMBED_DIMS = [64, 128, 256]
NUM_HEADS = [2, 4, 8]
DEPTH = [2, 2, 2]


# DATASET LOADER
class PhaseRetrievalDataset(Dataset):
    def __init__(self, data_path, illumination_type='standard'):
        self.gt_path = os.path.join(data_path, 'ground_truth')
        self.input_path = os.path.join(data_path, f'input_{illumination_type}')
        self.filenames = [f for f in os.listdir(self.gt_path) if f.endswith('.npy')]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        gt = np.load(os.path.join(self.gt_path, self.filenames[idx]))
        ipt = np.load(os.path.join(self.input_path, self.filenames[idx]))
        gt_tensor = torch.from_numpy(gt).unsqueeze(0).float()
        ipt_tensor = torch.from_numpy(ipt).unsqueeze(0).float()
        
        return ipt_tensor, gt_tensor

In [2]:
if __name__ == '__main__':
    print("Starting Fourier Transformer Debug Training Script")
    print(f"Using device: {DEVICE}")

    # Load debug data
    print(f"Loading data from: {DEBUG_DATA_PATH}")
    debug_dataset = PhaseRetrievalDataset(DEBUG_DATA_PATH, ILLUMINATION)
    debug_loader = DataLoader(debug_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    
    print(f"Found {len(debug_dataset)} images in the debug set.")

    # Initialize model
    model = FourierTransformerUNet(
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
        embed_dims=EMBED_DIMS,
        num_heads=NUM_HEADS,
        depth=DEPTH
    ).to(DEVICE)
    
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model initialized with {num_params / 1e6:.2f} M parameters.")

    # Loss function and optimizer
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("\nStarting training loop")
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for inputs, targets in tqdm(debug_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            predictions = model(inputs)
            loss = loss_fn(predictions, targets)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_epoch_loss = epoch_loss / len(debug_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} -> Average Loss: {avg_epoch_loss:.6f}")

    print("\nDebug script finished successfully")

Starting Fourier Transformer Debug Training Script
Using device: cpu
Loading data from: ../data/processed/flickr30k_256x256_augmented\debug
Found 15 images in the debug set.
Model initialized with 4.45 M parameters.

Starting training loop


Epoch 1/8:   0%|                                                                                 | 0/3 [00:00<?, ?it/s]C:\Users\nitin\AppData\Roaming\Python\Python312\site-packages\torch\autograd\graph.py:824: UserWarning: Failed to initialize XPU devices. The driver may not be installed, installed incorrectly, or incompatible with the current setup. Please refer to the guideline (https://github.com/pytorch/pytorch?tab=readme-ov-file#intel-gpu-support) for proper installation and configuration. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\xpu\XPUFunctions.cpp:111.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Epoch 1/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:08<00:00,  2.86s/it]


Epoch 1/8 -> Average Loss: 0.282416


Epoch 2/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:04<00:00,  1.66s/it]


Epoch 2/8 -> Average Loss: 0.131125


...

Epoch 6/8 -> Average Loss: 0.092674


Epoch 7/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:07<00:00,  2.61s/it]


Epoch 7/8 -> Average Loss: 0.089839


Epoch 8/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:06<00:00,  2.08s/it]

Epoch 8/8 -> Average Loss: 0.087842

Debug script finished successfully


In [4]:
if __name__ == '__main__':
    print("Starting Fourier Transformer Debug Training Script")
    print(f"Using device: {DEVICE}")

    # Load debug data
    print(f"Loading data from: {DEBUG_DATA_PATH}")
    debug_dataset = PhaseRetrievalDataset(DEBUG_DATA_PATH, ILLUMINATION)
    debug_loader = DataLoader(debug_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    
    print(f"Found {len(debug_dataset)} images in the debug set.")

    # Initialize model
    model = FourierTransformerUNet(
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
        embed_dims=EMBED_DIMS,
        num_heads=NUM_HEADS,
        depth=DEPTH
    ).to(DEVICE)
    
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model initialized with {num_params / 1e6:.2f} M parameters.")

    # Loss function and optimizer
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("\nStarting training loop")
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for inputs, targets in tqdm(debug_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            predictions = model(inputs)
            loss = loss_fn(predictions, targets)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_epoch_loss = epoch_loss / len(debug_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} -> Average Loss: {avg_epoch_loss:.6f}")

    print("\nDebug script finished successfully")

Starting Fourier Transformer Debug Training Script
Using device: cpu
Loading data from: ../data/processed/flickr30k_256x256_augmented\debug
Found 15 images in the debug set.
Model initialized with 4.45 M parameters.

Starting training loop


Epoch 1/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:04<00:00,  1.39s/it]


Epoch 1/8 -> Average Loss: 0.285453


Epoch 2/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:05<00:00,  1.88s/it]


Epoch 2/8 -> Average Loss: 0.149540


...

Epoch 6/8 -> Average Loss: 0.097970


Epoch 7/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:05<00:00,  1.89s/it]


Epoch 7/8 -> Average Loss: 0.094259


Epoch 8/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:06<00:00,  2.01s/it]

Epoch 8/8 -> Average Loss: 0.090825

Debug script finished successfully


In [7]:
if __name__ == '__main__':
    print("Starting Fourier Transformer Debug Training Script")
    print(f"Using device: {DEVICE}")

    # Load debug data
    print(f"Loading data from: {DEBUG_DATA_PATH}")
    debug_dataset = PhaseRetrievalDataset(DEBUG_DATA_PATH, ILLUMINATION)
    debug_loader = DataLoader(debug_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    
    print(f"Found {len(debug_dataset)} images in the debug set.")

    # Initialize model
    model = FourierTransformerUNet(
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
        embed_dims=EMBED_DIMS,
        depth=DEPTH
    ).to(DEVICE)
    
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model initialized with {num_params / 1e6:.2f} M parameters.")

    # Loss function and optimizer
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("\nStarting training loop")
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for inputs, targets in tqdm(debug_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            predictions = model(inputs)
            loss = loss_fn(predictions, targets)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_epoch_loss = epoch_loss / len(debug_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} -> Average Loss: {avg_epoch_loss:.6f}")

    print("\nDebug script finished successfully")

Starting Fourier Transformer Debug Training Script
Using device: cpu
Loading data from: ../data/processed/flickr30k_256x256_augmented\debug
Found 15 images in the debug set.
Model initialized with 3.53 M parameters.

Starting training loop


Epoch 1/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:06<00:00,  2.03s/it]


Epoch 1/8 -> Average Loss: 2.002718


Epoch 2/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:05<00:00,  1.73s/it]


Epoch 2/8 -> Average Loss: 1.553989


...

Epoch 6/8 -> Average Loss: 0.658959


Epoch 7/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:05<00:00,  1.71s/it]


Epoch 7/8 -> Average Loss: 0.549497


Epoch 8/8: 100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:05<00:00,  1.70s/it]

Epoch 8/8 -> Average Loss: 0.462379

Debug script finished successfully
